In [1]:
import pandas as pd

In [2]:
def clean_strings(df):
    string_cols = df.select_dtypes("object").columns
    df.loc[:, string_cols] = df.loc[:, string_cols].apply(lambda x: x.str.strip().str.replace(r"\s+", " "))
    return df


def drop_suffix_cols(df, suffix="_y"):
    return df.drop(columns=[col for col in df.columns if col.endswith(suffix)])


def merge_and_clean_df_2020_22(df_4, df_5, suffix="_y", drop_suffix=False):
    df = pd.merge(
        left=df_4,
        right=df_5,
        on=["Diarienummer"],
        how="left",
        suffixes=(None, suffix),
    )

    df = df.assign(
        Län=df[f"Län{suffix}"],
        Kommun=df[f"Kommun{suffix}"],
    )

    if drop_suffix:
        df = drop_suffix_cols(df)

    return df


def merge_and_clean_df_2023_24(df_4, df_5, suffix="_y", drop_suffix=False):
    df = pd.merge(
        left=df_4.loc[df_4["Flera kommuner"] == "Ja"],
        # left=df_4,
        right=df_5,
        on=["Diarienummer"],
        how="left",
        suffixes=(None, suffix),
    )

    df = df.assign(
        Län=df[f"Län{suffix}"],
        Kommun=df[f"Kommun{suffix}"],
    )

    df = pd.concat([df, df_4.loc[df_4["Flera kommuner"] != "Ja"]])

    if drop_suffix:
        df = drop_suffix_cols(df)

    return df

In [3]:
rename_cols = {"Flera studiekommuner": "Flera kommuner", "Antal studiekommuner": "Antal kommuner"}

df_2020_4 = pd.read_excel("data/resultat-ansokningsomgang-2020.xlsx", sheet_name=4)
df_2020_5 = pd.read_excel("data/resultat-ansokningsomgang-2020.xlsx", sheet_name=5)

df_2021_4 = pd.read_excel("data/resultat-ansokningsomgang-2021.xlsx", sheet_name=4)
df_2021_5 = pd.read_excel("data/resultat-ansokningsomgang-2021.xlsx", sheet_name=5)

df_2022_4 = pd.read_excel("data/resultat-ansokningsomgang-2022.xlsx", sheet_name=4)
df_2022_5 = pd.read_excel("data/resultat-ansokningsomgang-2022.xlsx", sheet_name=5)

dataframes_2020_2022 = [df_2020_4, df_2020_5, df_2021_4, df_2021_5, df_2022_4, df_2022_5]
for df in dataframes_2020_2022:
    df.rename(columns=rename_cols, inplace=True)

df_2023_4 = pd.read_excel("data/resultat-ansokningsomgang-2023.xlsx", sheet_name=4, skiprows=5)
df_2023_5 = pd.read_excel("data/resultat-ansokningsomgang-2023.xlsx", sheet_name=5, skiprows=5)

df_2024_4 = pd.read_excel("data/resultat-ansokningsomgang-2024.xlsx", sheet_name=4, skiprows=5)
df_2024_5 = pd.read_excel("data/resultat-ansokningsomgang-2024.xlsx", sheet_name=5, skiprows=5)

In [7]:
df_2020_0 = merge_and_clean_df_2020_22(df_2020_4, df_2020_5)
df_2020_0["Ansökningsomgång"] = 2020
df_2021_0 = merge_and_clean_df_2020_22(df_2021_4, df_2021_5)
df_2021_0["Ansökningsomgång"] = 2021
df_2022_0 = merge_and_clean_df_2020_22(df_2022_4, df_2022_5)
df_2022_0["Ansökningsomgång"] = 2022

df_2023_0 = merge_and_clean_df_2023_24(df_2023_4, df_2023_5)
df_2023_0["Ansökningsomgång"] = 2023
df_2024_0 = merge_and_clean_df_2023_24(df_2024_4, df_2024_5)
df_2024_0["Ansökningsomgång"] = 2024

annual_dataframes = [df_2020_0, df_2021_0, df_2022_0, df_2023_0, df_2024_0]
df = pd.concat(annual_dataframes)

cols_common = set.intersection(*[set(df.columns) for df in [*annual_dataframes]])
df = df[list(cols_common)]

df = drop_suffix_cols(df)

# Convert 'Flera kommuner' to bool
df["Flera kommuner"] = df["Flera kommuner"].map({"Ja": True, "Nej": False})
if df["Flera kommuner"].isna().sum() > 0:
    raise ValueError(f"Found {df['Flera kommuner'].isna().sum()} NaN values in 'Flera kommuner'")

# Standardization of 'Beslut' and convert to bool
df["Beslut"] = df["Beslut"].map({"Beviljad": True, "Ej beviljad": False, "Avslag": False})
if df["Beslut"].isna().sum() > 0:
    raise ValueError(f"Found {df['Beslut'].isna().sum()} NaN values in 'Beslut'")

df = clean_strings(df)

df = df.sort_values(by=["Ansökningsomgång", "Diarienummer", "Län", "Kommun"]).reset_index(drop=True)

cols_order = [
    "Diarienummer",
    "Ansökningsomgång",
    "Beslut",
    "Utbildningsanordnare administrativ enhet",
    "Huvudmannatyp",
    "Utbildningsområde",
    "Utbildningsnamn",
    "YH-poäng",
    "Studieform",
    "Studietakt %",
    "Sökta utbildningsomgångar",
    "Beviljade utbildningsomgångar",
    "Sökta platser totalt",
    "Beviljade platser totalt",
    "Sökta platser per utbildningsomgång",
    "Län",
    "Kommun",
    "Antal kommuner",
    "Flera kommuner",
]

df = df[cols_order].rename(columns={"Utbildningsanordnare administrativ enhet": "Utbildningsanordnare"})

df.to_parquet("data/resultat-ansokningsomgang-2020-2024-beslut.parquet", index=False)
df.to_csv("data/resultat-ansokningsomgang-2020-2024-beslut_join.csv", index=False)

df

,Diarienummer,Ansökningsomgång,Beslut,Utbildningsanordnare,Huvudmannatyp,Utbildningsområde,Utbildningsnamn,YH-poäng,Studieform,Studietakt %,Sökta utbildningsomgångar,Beviljade utbildningsomgångar,Sökta platser totalt,Beviljade platser totalt,Sökta platser per utbildningsomgång,Län,Kommun,Antal kommuner,Flera kommuner
0,MYH 2020/1698,2020,True,Folkuniversitetet – Kursverksamheten vid Umeå ...,Privat,Samhällsbyggnad och byggteknik,VA-projektör,400,Bunden,100,3,3,30,30.0,10,Gävleborg,Gävle,4,True
1,MYH 2020/1698,2020,True,Folkuniversitetet – Kursverksamheten vid Umeå ...,Privat,Samhällsbyggnad och byggteknik,VA-projektör,400,Bunden,100,3,3,9,9.0,3,Jämtland,Östersund,4,True
2,MYH 2020/1698,2020,True,Folkuniversitetet – Kursverksamheten vid Umeå ...,Privat,Samhällsbyggnad och byggteknik,VA-projektör,400,Bunden,100,3,3,15,15.0,5,Västerbotten,Umeå,4,True
3,MYH 2020/1698,2020,True,Folkuniversitetet – Kursverksamheten vid Umeå ...,Privat,Samhällsbyggnad och byggteknik,VA-projektör,400,Bunden,100,3,3,21,21.0,7,Västernorrland,Sundsvall,4,True
4,MYH 2020/3237,2020,True,Campus Nyköping,Kommun,Hälso- och sjukvård samt socialt arbete,Behandlingspedagog,400,Bunden,100,4,3,140,105.0,35,Södermanland,Nyköping,1,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8584,MYH 2024/4273,2024,False,AVIC AB,Privat,Teknik och tillverkning,Automationstekniker,400,Distans,100,3,0,75,0.0,25,Stockholm,Stockholm,1,False
8585,MYH 2024/4274,2024,True,Umeå kommun - Dragonskolan,Kommun,Samhällsbyggnad och byggteknik,"Byggnadsvårdsingenjör - arbetsledning, renover...",415,Distans,100,2,2,40,40.0,20,Västerbotten,Umeå,1,False
8586,MYH 2024/4275,2024,False,AVIC AB,Privat,Samhällsbyggnad och byggteknik,Optimering av hållbara energisystem med AI-stöd,400,Distans,100,4,0,104,0.0,26,Stockholm,Stockholm,1,False
8587,MYH 2024/4276,2024,False,IT-Högskolan Sverige AB,Privat,Data/IT,Python-Engineer,400,Bunden,100,3,0,105,0.0,35,Västra Götaland,Göteborg,1,False
